# Assignment 7 — Delta Lake MERGE (PySpark / Databricks version)

This is the **Spark** implementation of the same pipeline as
`delta_scd_assignment.ipynb`. Same data, same cleaning rules, same `MERGE` semantics,
same results — expressed with `pyspark` + `delta-spark` instead of `delta-rs`.

**Where to run this**

| Environment | What to do |
|---|---|
| **Databricks** (Community Edition works) | Import the notebook, upload the two CSVs to DBFS/Volumes, set `DATA_DIR`, skip the "build a SparkSession" cell — the cluster already has one |
| **Google Colab** | Run the install cell — it fetches Java and the Delta jars from Maven |
| **Local machine** | Needs Java 8/11/17 on `PATH` plus outbound access to Maven Central for the Delta jars |

> The `delta-rs` notebook (`delta_scd_assignment.ipynb`) is the primary submission because it
> needs no JVM and no Maven download, so it runs anywhere. Both write the identical Delta
> transaction log — a table created by one is fully readable by the other.

In [ ]:
# Colab / local only. On Databricks skip this cell entirely.
# !apt-get -qq install openjdk-17-jdk-headless > /dev/null
# !pip install -q pyspark==3.5.1 delta-spark==3.2.0
# import os; os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType)
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# On Databricks a SparkSession already exists - this whole block can be skipped.
builder = (
    SparkSession.builder.appName("Assignment7-DeltaLake-MERGE")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("spark version :", spark.version)
print("delta enabled :",
      spark.conf.get("spark.sql.extensions") == "io.delta.sql.DeltaSparkSessionExtension")

In [ ]:
import os
from pathlib import Path

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_DIR = PROJECT_ROOT / "data"          # on Databricks: "/Volumes/.../data" or "dbfs:/FileStore/..."
LAKE_DIR = PROJECT_ROOT / "output" / "delta_lake_spark"

MASTER_CSV = str(DATA_DIR / "customer_master.csv")
INCR_CSV   = str(DATA_DIR / "customer_incremental.csv")

TBL_BRONZE = str(LAKE_DIR / "bronze_customer_raw")
TBL_SILVER = str(LAKE_DIR / "silver_customer_master")
TBL_SCD1   = str(LAKE_DIR / "gold_customer_scd1")
TBL_SCD2   = str(LAKE_DIR / "gold_customer_scd2")

BUSINESS_KEY = "customer_id"
SNAPSHOT_TS  = "2019-12-31 22:00:00"
BATCH_TS     = "2020-01-15 06:30:00"

TRACKED_ATTRS = ["segment", "city", "state", "postal_code", "region", "loyalty_tier",
                 "email", "total_orders", "total_sales", "total_profit", "last_order_date"]

print("data dir :", DATA_DIR)
print("lake dir :", LAKE_DIR)

---
## 1. Load the raw snapshot into a Delta table (bronze)

Read everything as `STRING` first — the bronze layer lands the source verbatim so nothing is
silently coerced or lost (leading zeros in postal codes, for example).

In [ ]:
COLUMNS = ["customer_id", "customer_name", "segment", "country", "city", "state",
           "postal_code", "region", "total_orders", "total_sales", "total_profit",
           "last_order_date", "loyalty_tier", "email", "record_updated_at"]

raw_schema = StructType([StructField(c, StringType(), True) for c in COLUMNS])

master_raw = spark.read.csv(MASTER_CSV, header=True, schema=raw_schema)
incr_raw   = spark.read.csv(INCR_CSV,   header=True, schema=raw_schema)

print(f"master rows : {master_raw.count():,}")
print(f"incr   rows : {incr_raw.count():,}")
master_raw.show(5, truncate=False)

In [ ]:
# data quality profile BEFORE cleaning
total = master_raw.count()
null_counts = master_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in COLUMNS
]).collect()[0].asDict()

print(f"rows                        : {total:,}")
print(f"exact duplicate rows        : {total - master_raw.distinct().count():,}")
print(f"distinct customer_id        : {master_raw.select(BUSINESS_KEY).distinct().count():,}")
print("\nnulls per column:")
for c, n in null_counts.items():
    if n:
        print(f"  {c:<20} {n:>5}  ({n / total * 100:5.2f}%)")

In [ ]:
(master_raw.write.format("delta").mode("overwrite").save(TBL_BRONZE))

bronze = DeltaTable.forPath(spark, TBL_BRONZE)
print("bronze table written")
bronze.toDF().printSchema()
bronze.history().select("version", "operation", "operationMetrics").show(truncate=False)

---
## 2. Cleaning

The same seven rules as the primary notebook, expressed with Spark column expressions.
Step 5 — one row per business key — uses a window function, which is the Spark idiom for
"keep the newest record per key".

In [ ]:
def clean_customer_df(df, label="dataset"):
    rows_in = df.count()

    # 1/2  trim, collapse whitespace, standardise case, empty string -> NULL
    for c in df.columns:
        df = df.withColumn(c, F.trim(F.col(c)))
        df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
    df = (df
          .withColumn("customer_name", F.regexp_replace("customer_name", r"\s+", " "))
          .withColumn("segment",      F.initcap("segment"))
          .withColumn("region",       F.initcap("region"))
          .withColumn("city",         F.initcap("city"))
          .withColumn("loyalty_tier", F.upper("loyalty_tier"))
          .withColumn("email",        F.lower("email")))

    # 3  unusable business key
    df = df.filter(F.col(BUSINESS_KEY).isNotNull())
    after_key = df.count()

    # 4  exact duplicate rows
    df = df.dropDuplicates()
    after_exact = df.count()

    # 5  one row per business key - newest record_updated_at wins
    w = Window.partitionBy(BUSINESS_KEY).orderBy(F.col("record_updated_at").desc_nulls_last())
    df = (df.withColumn("_rn", F.row_number().over(w))
            .filter(F.col("_rn") == 1)
            .drop("_rn"))
    after_key_dedupe = df.count()

    # 6  types
    df = (df
          .withColumn("total_orders",  F.coalesce(F.col("total_orders").cast(IntegerType()), F.lit(0)))
          .withColumn("total_sales",   F.round(F.col("total_sales").cast(DoubleType()), 2))
          .withColumn("total_profit",  F.round(F.col("total_profit").cast(DoubleType()), 2))
          .withColumn("last_order_date",   F.to_date("last_order_date", "yyyy-MM-dd"))
          .withColumn("record_updated_at", F.to_timestamp("record_updated_at", "yyyy-MM-dd HH:mm:ss")))

    # 7  impute
    df = (df
          .fillna({"segment": "Unknown", "city": "Unknown", "state": "Unknown",
                   "region": "Unknown", "postal_code": "00000",
                   "country": "United States", "total_sales": 0.0, "total_profit": 0.0})
          .withColumn("loyalty_tier", F.coalesce(
              F.col("loyalty_tier"),
              F.when(F.col("total_sales") >= 5000, "PLATINUM")
               .when(F.col("total_sales") >= 2500, "GOLD")
               .when(F.col("total_sales") >= 1000, "SILVER")
               .otherwise("BRONZE")))
          .withColumn("email", F.coalesce(
              F.col("email"),
              F.concat(
                  F.regexp_replace(F.lower(F.trim(F.col("customer_name"))), r"[^a-z ]", ""),
                  F.lit("@unknown.local")))))
    df = df.withColumn("email", F.regexp_replace("email", " ", "."))

    print(f"--- cleaning report: {label} ---")
    print(f"  rows_in                  : {rows_in:,}")
    print(f"  dropped_null_key         : {rows_in - after_key:,}")
    print(f"  dropped_exact_duplicates : {after_key - after_exact:,}")
    print(f"  dropped_duplicate_keys   : {after_exact - after_key_dedupe:,}")
    print(f"  rows_out                 : {after_key_dedupe:,}")
    return df.orderBy(BUSINESS_KEY)


master_clean = clean_customer_df(master_raw, "customer_master.csv").cache()
incr_clean   = clean_customer_df(incr_raw,   "customer_incremental.csv").cache()
master_clean.show(5, truncate=False)

In [ ]:
# post-cleaning assertions
assert master_clean.filter(F.col(BUSINESS_KEY).isNull()).count() == 0
assert master_clean.count() == master_clean.select(BUSINESS_KEY).distinct().count()
assert master_clean.count() == master_clean.distinct().count()
print("no null keys, no duplicate keys, no duplicate rows - OK")

master_clean.write.format("delta").mode("overwrite").save(TBL_SILVER)
print(f"silver table written: {spark.read.format('delta').load(TBL_SILVER).count():,} rows")

### 2.1 Classify the incoming batch

An MD5 hash over the tracked attributes tells us, before merging, exactly how many rows are
new / changed / unchanged. These become the expected values the validation step asserts on.

In [ ]:
def with_hash(df):
    return df.withColumn(
        "record_hash",
        F.md5(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in TRACKED_ATTRS])),
    )


master_hashed = with_hash(master_clean)
incr_hashed   = with_hash(incr_clean).cache()

classified = (
    incr_hashed.alias("s")
    .join(master_hashed.select(BUSINESS_KEY, F.col("record_hash").alias("target_hash")).alias("t"),
          on=BUSINESS_KEY, how="left")
    .withColumn("change_type",
                F.when(F.col("target_hash").isNull(), "NEW")
                 .when(F.col("target_hash") != F.col("record_hash"), "CHANGED")
                 .otherwise("UNCHANGED"))
).cache()

counts = {r["change_type"]: r["count"] for r in
          classified.groupBy("change_type").count().collect()}
EXPECTED_NEW       = counts.get("NEW", 0)
EXPECTED_CHANGED   = counts.get("CHANGED", 0)
EXPECTED_UNCHANGED = counts.get("UNCHANGED", 0)

print(f"NEW       : {EXPECTED_NEW:,}   -> will be INSERTED")
print(f"CHANGED   : {EXPECTED_CHANGED:,}   -> will be UPDATED / versioned")
print(f"UNCHANGED : {EXPECTED_UNCHANGED:,}   -> no-op")

---
## 3. `MERGE` — SCD Type 1

One row per customer: matched rows are overwritten, unmatched rows are inserted.

```sql
MERGE INTO gold_customer_scd1 AS t
USING incremental_batch        AS s
   ON t.customer_id = s.customer_id
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

In [ ]:
# initial load
master_clean.write.format("delta").mode("overwrite").save(TBL_SCD1)
rows_before = spark.read.format("delta").load(TBL_SCD1).count()

scd1 = DeltaTable.forPath(spark, TBL_SCD1)

(scd1.alias("t")
     .merge(incr_clean.alias("s"), f"t.{BUSINESS_KEY} = s.{BUSINESS_KEY}")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

rows_after = spark.read.format("delta").load(TBL_SCD1).count()
print(f"{rows_before:,} rows -> {rows_after:,} rows")

metrics = (scd1.history(1).select("operationMetrics").collect()[0][0])
for k in ["numSourceRows", "numTargetRowsUpdated", "numTargetRowsInserted",
          "numTargetRowsDeleted", "numOutputRows"]:
    print(f"  {k:<24} : {metrics.get(k)}")

assert rows_after == rows_before + EXPECTED_NEW
assert int(metrics["numTargetRowsInserted"]) == EXPECTED_NEW
assert int(metrics["numTargetRowsUpdated"]) == EXPECTED_CHANGED + EXPECTED_UNCHANGED
print("SCD1 merge metrics match the classification exactly.")

In [ ]:
# the same MERGE written as SQL, for reference
spark.read.format("delta").load(TBL_SCD1).createOrReplaceTempView("v_scd1")
incr_clean.createOrReplaceTempView("v_incremental")

spark.sql(
    "SELECT touched, COUNT(*) AS customers FROM ("
    "  SELECT CASE WHEN last_order_date = DATE'2020-01-15'"
    "              THEN 'touched by this batch' ELSE 'untouched' END AS touched"
    "  FROM v_scd1"
    ") GROUP BY touched ORDER BY customers DESC"
).show()

# equivalent SQL MERGE (commented - the Python API above already ran it):
# MERGE INTO delta.`{TBL_SCD1}` AS t
# USING v_incremental AS s
#    ON t.customer_id = s.customer_id
# WHEN MATCHED     THEN UPDATE SET *
# WHEN NOT MATCHED THEN INSERT *

---
## 4. `MERGE` — SCD Type 2

Full history via `effective_start_date` / `effective_end_date` / `is_current`, using the
standard **two-part staged source**: Part A (keyed) closes the outgoing version, Part B
(`merge_key = NULL`, therefore never matched) inserts the incoming one.

```sql
MERGE INTO gold_customer_scd2 AS t
USING staged_source            AS s
   ON t.customer_id = s.merge_key AND t.is_current = true
WHEN MATCHED AND t.record_hash <> s.record_hash
     THEN UPDATE SET is_current = false, effective_end_date = s.effective_start_date
WHEN NOT MATCHED
     THEN INSERT *
```

In [ ]:
w_sk = Window.orderBy(BUSINESS_KEY)

scd2_seed = (master_hashed
             .withColumn("customer_sk", F.row_number().over(w_sk).cast("long"))
             .withColumn("effective_start_date", F.lit(SNAPSHOT_TS).cast("timestamp"))
             .withColumn("effective_end_date",   F.lit(None).cast("timestamp"))
             .withColumn("is_current",           F.lit(True)))

SCD2_COLUMNS = (["customer_sk"] + master_clean.columns +
                ["record_hash", "effective_start_date", "effective_end_date", "is_current"])
scd2_seed = scd2_seed.select(*SCD2_COLUMNS)

scd2_seed.write.format("delta").mode("overwrite").save(TBL_SCD2)
scd2_rows_before = spark.read.format("delta").load(TBL_SCD2).count()
print(f"SCD2 seeded with {scd2_rows_before:,} current rows")
scd2_seed.show(5, truncate=False)

In [ ]:
max_sk = spark.read.format("delta").load(TBL_SCD2).agg(F.max("customer_sk")).collect()[0][0]

src = (incr_hashed
       .withColumn("effective_start_date", F.lit(BATCH_TS).cast("timestamp"))
       .withColumn("effective_end_date",   F.lit(None).cast("timestamp"))
       .withColumn("is_current",           F.lit(True)))

changed_ids = (classified.filter(F.col("change_type") == "CHANGED")
                         .select(BUSINESS_KEY))

part_a = src.withColumn("merge_key", F.col(BUSINESS_KEY))
part_b = (src.join(F.broadcast(changed_ids), on=BUSINESS_KEY, how="inner")
             .withColumn("merge_key", F.lit(None).cast("string")))

staged = (part_a.select("merge_key", *[c for c in src.columns])
          .unionByName(part_b.select("merge_key", *[c for c in src.columns])))

w_new_sk = Window.orderBy("merge_key", BUSINESS_KEY)
staged = (staged
          .withColumn("customer_sk", (F.lit(max_sk) + F.row_number().over(w_new_sk)).cast("long"))
          .select("merge_key", *SCD2_COLUMNS))

print(f"Part A (all rows, keyed)        : {part_a.count():,}")
print(f"Part B (changed rows, NULL key) : {part_b.count():,}")
print(f"staged source                   : {staged.count():,}")
staged.filter(F.col(BUSINESS_KEY).isin([r[0] for r in changed_ids.limit(1).collect()])).show(truncate=False)

In [ ]:
scd2 = DeltaTable.forPath(spark, TBL_SCD2)

(scd2.alias("t")
     .merge(staged.alias("s"),
            f"t.{BUSINESS_KEY} = s.merge_key AND t.is_current = true")
     .whenMatchedUpdate(
         condition="t.record_hash <> s.record_hash",
         set={"is_current": F.lit(False),
              "effective_end_date": F.col("s.effective_start_date")})
     .whenNotMatchedInsert(values={c: F.col(f"s.{c}") for c in SCD2_COLUMNS})
     .execute())

scd2_df = spark.read.format("delta").load(TBL_SCD2)
scd2_rows_after = scd2_df.count()

m2 = scd2.history(1).select("operationMetrics").collect()[0][0]
print(f"{scd2_rows_before:,} rows -> {scd2_rows_after:,} rows")
print(f"  versions closed  : {m2.get('numTargetRowsUpdated')}   [expected {EXPECTED_CHANGED}]")
print(f"  rows inserted    : {m2.get('numTargetRowsInserted')}   "
      f"[expected {EXPECTED_CHANGED} + {EXPECTED_NEW} = {EXPECTED_CHANGED + EXPECTED_NEW}]")

In [ ]:
# a customer that was versioned by this batch
sample_id = (scd2_df.filter(~F.col("is_current")).select(BUSINESS_KEY).limit(1).collect()[0][0])

(scd2_df.filter(F.col(BUSINESS_KEY) == sample_id)
        .orderBy("effective_start_date")
        .select("customer_sk", BUSINESS_KEY, "segment", "city", "total_sales",
                "effective_start_date", "effective_end_date", "is_current")
        .show(truncate=False))

---
## 5. Validation

In [ ]:
scd1_df = spark.read.format("delta").load(TBL_SCD1)
cur = scd2_df.filter(F.col("is_current"))

checks = [
    ("scd1: unique customer_id",
     scd1_df.count() == scd1_df.select(BUSINESS_KEY).distinct().count()),
    ("scd1: no duplicate rows",
     scd1_df.count() == scd1_df.distinct().count()),
    ("scd1: row count = seed + new",
     scd1_df.count() == rows_before + EXPECTED_NEW),
    ("scd2: unique surrogate key",
     scd2_df.count() == scd2_df.select("customer_sk").distinct().count()),
    ("scd2: one current row per customer",
     cur.count() == cur.select(BUSINESS_KEY).distinct().count()),
    ("scd2: current rows == scd1 rows",
     cur.count() == scd1_df.count()),
    ("scd2: current rows have NULL end date",
     cur.filter(F.col("effective_end_date").isNotNull()).count() == 0),
    ("scd2: expired rows have an end date",
     scd2_df.filter(~F.col("is_current") & F.col("effective_end_date").isNull()).count() == 0),
    ("scd2: expired count == changed count",
     scd2_df.filter(~F.col("is_current")).count() == EXPECTED_CHANGED),
]

for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(ok for _, ok in checks), "validation failed"
print(f"\n{len(checks)}/{len(checks)} checks passed")

In [ ]:
# transaction log + time travel
for name, path in [("scd1", TBL_SCD1), ("scd2", TBL_SCD2)]:
    print(f"\n--- {name} history ---")
    (DeltaTable.forPath(spark, path).history()
        .select("version", "timestamp", "operation")
        .orderBy("version").show(truncate=False))

v0 = spark.read.format("delta").option("versionAsOf", 0).load(TBL_SCD1)
v1 = spark.read.format("delta").option("versionAsOf", 1).load(TBL_SCD1)
print(f"time travel: v0 = {v0.count():,} rows, v1 = {v1.count():,} rows "
      f"(+{v1.count() - v0.count():,})")

---
## 6. Final dataset and summary

In [ ]:
print("top 10 customers by lifetime sales:")
(scd1_df.orderBy(F.col("total_sales").desc())
        .select(BUSINESS_KEY, "customer_name", "segment", "city", "region",
                "total_orders", "total_sales", "loyalty_tier")
        .show(10, truncate=False))

print("summary by region:")
(scd1_df.groupBy("region")
        .agg(F.count("*").alias("customers"),
             F.round(F.sum("total_sales"), 2).alias("sales"),
             F.round(F.sum("total_profit"), 2).alias("profit"))
        .orderBy(F.col("sales").desc())
        .show(truncate=False))

print("summary by loyalty tier:")
(scd1_df.groupBy("loyalty_tier")
        .agg(F.count("*").alias("customers"),
             F.round(F.sum("total_sales"), 2).alias("sales"))
        .orderBy(F.col("sales").desc())
        .show(truncate=False))

print(f"total customers : {scd1_df.count():,}")
print(f"total sales     : {scd1_df.agg(F.sum('total_sales')).collect()[0][0]:,.2f}")

In [ ]:
# spark.stop()   # uncomment when running locally and you are done
print("Pipeline complete — bronze, silver, SCD1 and SCD2 Delta tables written to", LAKE_DIR)

---
## Notes on the Spark implementation

* `whenMatchedUpdateAll()` / `whenNotMatchedInsertAll()` are the Python equivalents of
  `UPDATE SET *` / `INSERT *`. They require the source to have the same column names as the
  target, which is why the SCD2 staged source is projected onto `SCD2_COLUMNS` explicitly.
* The source **must** be de-duplicated on the merge key. If two source rows match the same
  target row, Delta raises
  `UnsupportedOperationException: Cannot perform Merge as multiple source rows matched...`.
* `Window.orderBy(...)` without a `partitionBy` (used here for surrogate keys) funnels all rows
  through a single partition. That is fine at this scale; at production volume use
  `monotonically_increasing_id()` or an identity column instead.
* On Databricks, replace the path-based tables with managed tables
  (`saveAsTable("catalog.schema.gold_customer_scd1")`) and the `DESCRIBE HISTORY`,
  `OPTIMIZE`, `ZORDER BY` and `VACUUM` commands all become available in SQL.